# Residual Connection (잔차 연결)

층이 깊어질수록 학습이 불안정해지는 문제(기울기 소실)를 완화하기 위해, 입력을 Sublayer의 출력에 그대로 더해준다.

수식은 아주 단순하다.

**output = x + Sublayer(x)**  (Sublayer는 Attention이나 Feed Forward 같은 하위 연산)

* Sublayer가 아무것도 학습하지 못해도 최소한 입력 x는 그대로 다음 층에 전달됨 (identity 보장) → 학습이 훨씬 안정적
* 이미지 분야의 ResNet에서 먼저 검증된 아이디어를 Transformer가 그대로 차용한 것

# Layer Normalization

**"한 샘플, 한 위치(토큰)" 안에서** 평균 0, 분산 1이 되도록 정규화한 뒤, 학습 가능한 파라미터로 다시 스케일/시프트한다.

LN(x) = γ · (x − μ) / √(σ² + ε) + β

* **μ, σ²**: 하나의 토큰 벡터(feature 차원) 안에서 계산한 평균과 분산
* **γ, β**: 학습되는 스케일/시프트 파라미터 (정규화 후 다시 표현력을 되찾아줌)
* **ε**: 분모가 0이 되는 것을 막는 아주 작은 상수

### Layer Norm vs Batch Norm — 왜 NLP는 Batch Norm 대신 Layer Norm을 쓰는가

| | Batch Norm | Layer Norm |
|---|---|---|
| 정규화 방향 | 배치(batch) 방향 | 한 토큰의 feature 방향 |
| 배치 크기 영향 | 배치가 작으면 통계가 불안정 | 배치 크기와 무관하게 항상 안정적 |
| 가변 길이 시퀀스 | 문장마다 길이가 달라 패딩 위치까지 섞이면 왜곡됨 | 각 토큰이 독립적으로 정규화되어 자연스러움 |

In [1]:
import torch
import torch.nn as nn

# 임의로 평균/분산이 큰 입력을 만들어 LayerNorm 전후를 비교
x = torch.randn(2, 4, 8) * 5 + 3   # [batch=2, seq_len=4, d_model=8]
layer_norm = nn.LayerNorm(8)
normed = layer_norm(x)

print("정규화 전 - 평균:", x.mean().item(), "표준편차:", x.std().item())
print("정규화 후 - 평균:", normed.mean().item(), "표준편차:", normed.std().item())

정규화 전 - 평균: 3.751354217529297 표준편차: 3.9612417221069336
정규화 후 - 평균: 1.4901161193847656e-08 표준편차: 1.0079048871994019


# Position-wise Feed Forward Network

각 단어 위치마다 **독립적으로** 적용되는 2층 MLP.

FFN(x) = max(0, xW₁ + b₁) W₂ + b₂  →  Linear → ReLU → Linear

* "position-wise"라는 이름대로, **같은 FFN을 모든 위치(단어)에 각각 독립적으로** 적용 (위치끼리 정보 교환은 없음)
* Attention이 "단어 사이" 정보를 섞는 역할이라면, FFN은 "섞인 정보를 각 위치에서 한 번 더 가공"하는 역할
* 보통 은닉 차원(d_ff)을 d_model보다 훨씬 크게(예: 512 → 2048) 키웠다가 다시 줄여서 표현력을 확보 (좁게 들어와서, 넓어졌다가, 다시 좁아지는 구조)

In [2]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)   # [B, T, d_model] → 동일 shape

In [3]:
# 입력과 출력의 shape이 동일한지 확인
ffn = PositionwiseFeedForward(d_model=128, d_ff=512)
x = torch.randn(2, 10, 128)
print(ffn(x).shape)   # torch.Size([2, 10, 128])

torch.Size([2, 10, 128])


# Encoder Layer

Multi-Head Attention과 Feed Forward, 각각에 **Add & Norm(잔차 연결 + Layer Normalization)**을 붙이면 Encoder Layer 하나가 완성된다.

입력 → Multi-Head Attention → Add & Norm → Feed Forward → Add & Norm → 출력

**Dropout은 어디에 넣는가**
* Attention weights 직후, FFN 출력 직후, 그리고 Add(잔차 연결) 직전에 Dropout을 적용하는 것이 관례
* 학습 초반에 과적합 걱정이 없다면 0.1 정도의 작은 값으로 시작해도 충분

### 이전에 구현한 PositionalEncoding, MultiHead Attention 복습

Encoder Layer는 이 둘을 그대로 활용해서 조립하므로 다시 가져온다.

In [4]:
import math

# 3주차 ~ 4주차에서 구현한 Positional Encoding 그대로 재사용
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        T = x.size(1)

        return x + self.pe[:, :T]

In [5]:
# 들어오는 Q, K, V는 W_q, W_k, W_v와 Embedding 결과 값을 각각 계산한 값.
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    weights = torch.softmax(scores, dim=-1)
    output = weights @ V

    return output, weights

# MultiHead Attention Layer 구현 (4주차와 동일)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.h = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, _ = x.shape
        x = x.view(B, T, self.h, self.d_k)

        return x.transpose(1, 2)  # [B, h, T, d_k]

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        out, weights = scaled_dot_product_attention(Q, K, V, mask)

        out = out.transpose(1, 2).contiguous().view(B, T, -1)

        return self.W_o(out)

In [6]:
# EncoderLayer: MHA + Add&Norm + FFN + Add&Norm
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # 1) Multi-Head Attention + 잔차 연결 + LayerNorm
        attn_out = self.mha(x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        # 2) Feed Forward + 잔차 연결 + LayerNorm
        # x + dropout(sublayer(x)) 패턴이 핵심 — 이 한 줄에 Residual Connection, Dropout, Sublayer가 모두 들어있음.
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x

In [7]:
# 더미 입력으로 shape이 그대로 유지되는지 확인
encoder_layer = EncoderLayer(d_model=128, num_heads=8, d_ff=512)
x = torch.randn(2, 10, 128)
print(encoder_layer(x).shape)   # torch.Size([2, 10, 128]) — 입력과 동일!

torch.Size([2, 10, 128])


# Encoder Stack

EncoderLayer를 N개 쌓으면 Encoder 전체가 된다. Encoder는 입력과 출력의 **shape이 항상 동일**([B, T, d_model] → [B, T, d_model])하기 때문에, EncoderLayer를 몇 개든 자유롭게 쌓을 수 있다.

In [8]:
class Encoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)   # 이전 layer의 출력이 다음 layer의 입력

        return x

In [9]:
x = torch.randn(2, 10, 512)   # [batch=2, seq_len=10, d_model=512]
encoder = Encoder(num_layers=6, d_model=512, num_heads=8, d_ff=2048)

out = encoder(x)
print(out.shape)   # torch.Size([2, 10, 512]) — 입력과 동일!

torch.Size([2, 10, 512])


# FULL Code

Raw Data -> Tokenizer -> Padding or Truncate -> Embedding + Positional Encoding -> **Encoder(Multi-Head Attention + Feed Forward, N Layers)**

In [10]:
# 예시로 감정 분류 모델을 Encoder Stack까지 확장시켜보면 아래와 같아짐.
class TSModel(nn.Module):
    def __init__(self, vocab_size=50000, embed_dim=128, num_classes=2, num_layers=2, num_heads=8, d_ff=512):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

        # Embedding
        self.embed_layer = nn.Embedding(vocab_size, embed_dim)

        # Positional Encoding
        self.pe_layer = PositionalEncoding(d_model=embed_dim)

        # 🍏🍏
        # Encoder Stack (Multi-Head Attention + Feed Forward를 num_layers번 반복)
        self.encoder = Encoder(num_layers=num_layers, d_model=embed_dim, num_heads=num_heads, d_ff=d_ff)

        # 감정 분류(클래스가 2개)이므로 마지막 feed forward만 넣어주면 우리가 원하는 모델을 만들 수 있음.
        self.feed_forward = nn.Linear(embed_dim, num_classes)

        # softmax
        self.act = nn.Softmax(dim=-1)

    def forward(self, x):
        # Embedding
        embeded_x = self.embed_layer(x)

        # Positional Encoding
        pe_embeded_x = self.pe_layer(embeded_x)
        print(pe_embeded_x.shape)

        # 🍏🍏
        # Encoder Stack
        output = self.encoder(pe_embeded_x)
        print(output.shape)

        pooled = output.mean(dim=1)
        print(pooled.shape)

        # 긍정인지 부정인지 확인
        x = self.feed_forward(pooled)
        print(x.shape)

        # 확률값으로 변환
        x = self.act(x)

        return x

In [11]:
ts_model = TSModel()

In [12]:
x = torch.tensor([[1, 2, 3, 4]])
probas = ts_model(x)
print(probas)

torch.Size([1, 4, 128])
torch.Size([1, 4, 128])
torch.Size([1, 128])
torch.Size([1, 2])
tensor([[0.5805, 0.4195]], grad_fn=<SoftmaxBackward0>)


# 감정 분류 학습 시켜보기

Naver에서 제공한 영화 리뷰 감정 분류 데이터셋(NSMC)
* 0이 부정
* 1이 긍정

### 0. Utils

In [13]:
# vocab 만드는 함수
# setences는 vocab을 만들 때 필요한 데이터
def create_vocab(sentences, special_tokens=["<pad>", "<sos>", "<eos>", "<unk>"]):
    words = sorted(set(w for s in sentences for w in s.split()))
    vocab = special_tokens + words  # 특수 토큰들에 대해서도 index를 매기기 위해서 SPECIALS와 words를 합침.
    stoi = {w: i for i, w in enumerate(vocab)}
    itos = {i: w for i, w in enumerate(vocab)}

    return words, vocab, stoi, itos

In [14]:
import torch

# 앞에서 encode 함수는 string에 대해서 처리
def encode(sentence, stoi):
    # 예외처리가 동작하면서 이전에 없던 값들이면 <unk>로 대체.
    ids = [stoi.get(w, stoi["<unk>"]) for w in sentence.split()]
    return [stoi["<sos>"]] + ids + [stoi["<eos>"]]

# 실제로는 리스트 형태로 들어옴(텐서)
# 이에 맞춰서 preprocessing 함수를 새로 선언
def preprocessing(text, stoi, padding_size=10, padding_token=0):
    input_tokens = []
    for sentence in text:
        # 우선 각 토큰들을 encoding
        encoded_sentence = encode(sentence, stoi)

        # embedding에 들어갈 때는 모든 데이터의 크기가 동일해야 하므로, padding적용
        encoded_padded_sentence = encoded_sentence + [padding_token] * (padding_size - len(encoded_sentence))

        input_tokens.append(encoded_padded_sentence)

    return torch.tensor(input_tokens)  # tensor 적용은 반드시 해야함.

### 1. Load Data

In [15]:
import pandas as pd

train_url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
test_url  = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"

train_df = pd.read_csv(train_url, sep="\t")
test_df  = pd.read_csv(test_url, sep="\t")

print(train_df.head())

         id                                           document  label
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                                  너무재밓었다그래서보는것을추천한다      0
3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1


In [16]:
# NAN값 제거
train_df = train_df.dropna(subset=["document"])
test_df  = test_df.dropna(subset=["document"])

train_texts, train_labels = train_df["document"].tolist(), train_df["label"].tolist()
test_texts, test_labels   = test_df["document"].tolist(), test_df["label"].tolist()

### 2. Vocab 생성

In [17]:
# vocab 사전을 만들기 위한 데이터
sentences = train_texts

# 특수 토큰: 패딩, 문장 시작/끝, 미등록 단어
special_tokens = ["<pad>", "<sos>", "<eos>", "<unk>"]

words, vocab, stoi, itos = create_vocab(sentences[:10000], special_tokens)

### 3. Model 생성

In [ ]:
# 예시로 감정 분류 모델을 Encoder Stack까지 확장시켜보면 아래와 같아짐.
class TSModel(nn.Module):
    def __init__(self, vocab_size=50000, embed_dim=128, num_classes=2, num_layers=2, num_heads=8, d_ff=512):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

        # Embedding
        self.embed_layer = nn.Embedding(vocab_size, embed_dim)

        # Positional Encoding
        self.pe_layer = PositionalEncoding(d_model=embed_dim)

        # 🍏🍏
        # Encoder Stack (Multi-Head Attention + Feed Forward를 num_layers번 반복)
        self.encoder = Encoder(num_layers=num_layers, d_model=embed_dim, num_heads=num_heads, d_ff=d_ff)

        # 감정 분류(클래스가 2개)이므로 마지막 feed forward만 넣어주면 우리가 원하는 모델을 만들 수 있음.
        self.feed_forward = nn.Linear(embed_dim, num_classes)

        # softmax
        self.act = nn.Softmax(dim=-1)

    def forward(self, x):
        # Embedding
        embeded_x = self.embed_layer(x)

        # Positional Encoding
        pe_embeded_x = self.pe_layer(embeded_x)

        # 🍏🍏
        # Encoder Stack
        output = self.encoder(pe_embeded_x)

        pooled = output.mean(dim=1)

        # 긍정인지 부정인지 확인
        x = self.feed_forward(pooled)

        # 확률값으로 변환
        x = self.act(x)

        return x

In [19]:
# 감정 분류 모델 생성
set_model = TSModel(vocab_size=len(vocab))

### 4. DataSet과 DataLoader 생성

In [20]:
from torch.utils.data import Dataset, DataLoader

# 여기서 transform은 preprocessing을 의미
class NSMC_Dataset(Dataset):
    def __init__(self, texts, labels, stoi=None, transform=None):
        self.texts = texts
        self.labels = labels
        self.stoi = stoi
        self.transform = transform

    def __len__(self):
        # 원래 tensor로 들어오면 shape이나 size 함수로 선언 가능하지만, 리스트가 들어오므로 len으로 호출
        return len(self.texts)

    def __getitem__(self, idx):
        batch_texts = self.texts[idx]
        batch_labels = self.labels[idx]

        if self.transform:
            # preprocessing은 문장 리스트를 받아 [N, padding_size] 텐서를 반환하므로
            # 문장 1개도 리스트로 감싸서 넣고, 결과에서 배치 차원([0])을 벗겨 [padding_size]로 되돌림.
            batch_texts = self.transform([batch_texts], stoi, padding_size=50)[0]

        return batch_texts, batch_labels

In [21]:
train_dataset = NSMC_Dataset(train_texts, train_labels, stoi, preprocessing)
test_dataset = NSMC_Dataset(test_texts, test_labels, stoi, preprocessing)

In [22]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [23]:
next(iter(train_dataloader))[0].shape

torch.Size([32, 50])

In [24]:
output = set_model(next(iter(train_dataloader))[0])

### 5. 학습

In [25]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

# GPU 사용 가능하면 GPU, 아니면 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 위에서 이미 만들어둔 것들을 그대로 사용
# - set_model: TSModel(vocab_size=50000, embed_dim=128, num_classes=2, num_layers=2, ...)
# - train_dataloader, test_dataloader: NSMC 데이터를 stoi로 인코딩한 DataLoader
set_model = set_model.to(device)

# set_model의 forward가 마지막에 Softmax(dim=-1)로 "확률"을 반환하므로,
# 내부에서 log_softmax를 또 적용하는 CrossEntropyLoss 대신 NLLLoss(log(확률))를 사용.
criterion = nn.NLLLoss()
optimizer = torch.optim.Adam(set_model.parameters(), lr=1e-3)

# tqdm 갱신 주기 (20 batch마다 한 번씩만 postfix 갱신)
LOG_INTERVAL = 20

# Train 및 Eval 로직
num_epochs = 5
for epoch in range(num_epochs):
    # 1. model train
    set_model.train()
    train_loss = 0.0
    train_preds_all, train_labels_all = [], []

    with tqdm(train_dataloader, desc=f"[Epoch {epoch+1}/{num_epochs}] Train", leave=False) as train_bar:
        for batch_idx, (texts, labels) in enumerate(train_bar):
            texts, labels = texts.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            # 1-1. model 예측
            # set_model(texts)는 forward 내부에서 이미 시퀀스 축을 pooling해서
            # [B, num_classes] 확률을 반환함.
            probs = set_model(texts)  # [B, num_classes]

            # 1-2. loss function 계산
            loss = criterion(torch.log(probs.clamp_min(1e-9)), labels)

            # 1-3. 이전 기울기 초기화
            optimizer.zero_grad()

            # 1-4. 역전파(backpropagation) -> 기울기 계산
            loss.backward()

            # 1-5. 파라미터 업데이트
            optimizer.step()

            train_loss += loss.item() * texts.size(0)
            train_preds_all.extend(probs.argmax(dim=1).cpu().tolist())
            train_labels_all.extend(labels.cpu().tolist())

            # 20 batch마다 한 번씩 loss/accuracy 갱신
            if (batch_idx + 1) % LOG_INTERVAL == 0:
                train_accuracy = accuracy_score(train_labels_all, train_preds_all)
                train_bar.set_postfix(loss=f"{loss.item():.4f}", accuracy=f"{train_accuracy:.2%}")

    train_loss /= len(train_dataloader.dataset)
    train_accuracy = accuracy_score(train_labels_all, train_preds_all)

    # 2. model eval
    set_model.eval()
    eval_preds_all, eval_labels_all = [], []
    accuracy = 0.0

    with torch.no_grad(), tqdm(test_dataloader, desc=f"[Epoch {epoch+1}/{num_epochs}] Eval", leave=False) as eval_bar:
        for batch_idx, (texts, labels) in enumerate(eval_bar):
            texts, labels = texts.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            probs = set_model(texts)  # [B, num_classes]
            eval_preds_all.extend(probs.argmax(dim=1).cpu().tolist())
            eval_labels_all.extend(labels.cpu().tolist())

            # 20 batch마다 한 번씩 accuracy 갱신
            if (batch_idx + 1) % LOG_INTERVAL == 0:
                accuracy = accuracy_score(eval_labels_all, eval_preds_all)
                eval_bar.set_postfix(accuracy=f"{accuracy:.2%}")

    accuracy = accuracy_score(eval_labels_all, eval_preds_all)

    print(f"epoch {epoch+1}/{num_epochs} | train_loss {train_loss:.4f} | train_accuracy {train_accuracy:.2%} | accuracy {accuracy:.2%}")

# 모델 저장
model_path = "ts_model_encoder_nsmc.pth"
torch.save(set_model.state_dict(), model_path)
print(f"모델 저장 완료: {model_path}")

Using device: cuda


[Epoch 1/5] Train:   0%|          | 0/4688 [00:00<?, ?it/s]

[Epoch 1/5] Eval:   0%|          | 0/1563 [00:00<?, ?it/s]

epoch 1/5 | train_loss 0.5592 | train_accuracy 69.04% | accuracy 73.45%


[Epoch 2/5] Train:   0%|          | 0/4688 [00:00<?, ?it/s]

[Epoch 2/5] Eval:   0%|          | 0/1563 [00:00<?, ?it/s]

... (6 outputs left out)

epoch 4/5 | train_loss 0.4015 | train_accuracy 79.83% | accuracy 75.11%


[Epoch 5/5] Train:   0%|          | 0/4688 [00:00<?, ?it/s]

[Epoch 5/5] Eval:   0%|          | 0/1563 [00:00<?, ?it/s]

epoch 5/5 | train_loss 0.3828 | train_accuracy 80.89% | accuracy 75.93%
모델 저장 완료: ts_model_encoder_nsmc.pth


### 6. 예측

실제 모델은 당연히 숫자값들에 대해 추론을 하지만, 여기서는 한국말이 나오면 그대로 나오도록 하는 함수를 작성

In [29]:
def predict(set_model, text, stoi, padding_size=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    preprocessed_text = preprocessing([text], stoi=stoi, padding_size=padding_size)
    preprocessed_text = preprocessed_text.to(device)

    probs = set_model(preprocessed_text)  # [1, num_classes]

    pred = probs.argmax(dim=-1).item()

    itos = ["부정", "긍정"]

    return itos[pred]

In [30]:
pred = predict(set_model, "이거 좋다.", stoi, padding_size=50)

In [31]:
pred

'긍정'

In [33]:
while True:
    text = input("🫠 텍스트를 입력하세요: ")
    if text == "exit":
        break
    print(f"User: {text}")

    pred = predict(set_model, text, stoi, padding_size=50)

    print(f">> 당신이 하시는 말씀은 {pred}적인 어투입니다 !")

User: 이거 싫어
>> 당신이 하시는 말씀은 부정적인 어투입니다 !
User: 이거 좋아.
>> 당신이 하시는 말씀은 긍정적인 어투입니다 !
User: 
>> 당신이 하시는 말씀은 긍정적인 어투입니다 !
